In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import json, torch

import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from PIL import Image
import tempfile

from torch.utils.data import Dataset, DataLoader
from transformers import (
    RTDetrForObjectDetection,
    RTDetrImageProcessor,
    TrainingArguments,
    Trainer,
)

import re

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from transformers.trainer_utils import get_last_checkpoint

In [5]:
!pip install -q transformers datasets accelerate torchvision timm

In [4]:
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [18]:
DATA_ROOT = Path('/kaggle/input/datasets/cybersimar08/drone-detection/coco json drone detection')
SPLITS    = ['train', 'valid', 'test']

In [8]:
def load_coco(split: str):
    split_dir  = DATA_ROOT / split
    json_files = list(split_dir.glob('*.json'))
    assert json_files, f'JSON не знайдено у {split_dir}'
    with open(json_files[0]) as f:
        return json.load(f), split_dir

In [9]:
data = {}
for split in SPLITS:
    try:
        coco, img_dir       = load_coco(split)
        data[split]         = {'coco': coco, 'img_dir': img_dir}
        print(f'[{split:>5}]  зображень: {len(coco["images"]):>5} | '
              f'анотацій: {len(coco["annotations"]):>6}')
    except AssertionError as e:
        print(f'[{split}]  {e}')

[train]  зображень: 10799 | анотацій:   8999
[valid]  зображень:   603 | анотацій:    497
[ test]  зображень:   596 | анотацій:    479


In [10]:
def build_df(coco: dict) -> pd.DataFrame:
    img_map = {img['id']: img for img in coco['images']}
    rows = []
    for ann in coco['annotations']:
        img      = img_map[ann['image_id']]
        x, y, w, h = ann['bbox']
        rows.append({
            'image_id' : ann['image_id'],
            'img_w'    : img['width'],
            'img_h'    : img['height'],
            'bbox_x'   : x,  'bbox_y': y,
            'bbox_w'   : w,  'bbox_h': h,
            'area'     : ann.get('area', w * h),
            'cx_rel'   : (x + w / 2) / img['width'],
            'cy_rel'   : (y + h / 2) / img['height'],
            'w_rel'    : w / img['width'],
            'h_rel'    : h / img['height'],
            'aspect'   : w / (h + 1e-6),
            'file_name': img['file_name'],
        })
    return pd.DataFrame(rows)

dfs    = {split: build_df(data[split]['coco']) for split in data}
df_all = (pd.concat(dfs.values(), keys=dfs.keys())
            .reset_index(level=0)
            .rename(columns={'level_0': 'split'}))

print('Статистика bbox')
df_all.groupby('split')[['bbox_w', 'bbox_h', 'area', 'aspect']].describe().round(1)

Статистика bbox


bbox_w                                             bbox_h        ...  \
        count  mean   std   min   25%   50%   75%    max   count  mean  ...   
split                                                                   ...   
test    479.0  54.9  42.9  16.0  29.5  38.5  63.2  275.5   479.0  57.2  ...   
train  8999.0  56.7  48.7   9.0  29.0  38.0  62.5  371.0  8999.0  56.1  ...   
valid   497.0  58.9  52.2  12.5  28.5  39.0  66.0  363.5   497.0  58.4  ...   

         area            aspect                                     
          75%       max   count mean  std  min  25%  50%  75%  max  
split                                                               
test   3818.5   94358.8   479.0  1.0  0.2  0.5  0.9  1.0  1.1  1.8  
train  3593.4  112112.0  8999.0  1.0  0.3  0.3  0.9  1.0  1.2  8.2  
valid  4168.8  112866.8   497.0  1.0  0.2  0.6  0.9  1.0  1.2  2.3  

[3 rows x 32 columns]

In [11]:
def extract_class_from_filename(filename: str) -> str:
    """
    V_DRONE_036146_...   - 'drone'
    V_AIRPLANE_0011_...  - 'airplane'
    V_HELICOPTER_047_... - 'helicopter'
    V_BIRD_01271_...     - 'bird'
    """
    match = re.match(r'V_([A-Z]+)_', Path(filename).name)
    if match:
        return match.group(1).lower()
    return 'unknown'

df_all['true_class'] = df_all['file_name'].apply(extract_class_from_filename)

print('Розподіл класів')
display(df_all.groupby(['split', 'true_class']).size()
              .unstack(fill_value=0)
              .assign(TOTAL=lambda d: d.sum(axis=1)))

Розподіл класів


true_class,airplane,bird,drone,helicopter,TOTAL
split,,,,,
test,128,0,237,114,479
train,2274,2,4349,2374,8999
valid,133,0,224,140,497


In [12]:
df_all_no_birds = df_all[df_all['true_class'] != 'bird']

In [13]:
print('Розподіл класів')
display(df_all_no_birds.groupby(['split', 'true_class']).size()
              .unstack(fill_value=0)
              .assign(TOTAL=lambda d: d.sum(axis=1)))

Розподіл класів


true_class,airplane,drone,helicopter,TOTAL
split,,,,
test,128,237,114,479
train,2274,4349,2374,8997
valid,133,224,140,497


In [19]:
OUTPUT_DIR = Path('rtdetr_drone_with_synt')
OUTPUT_DIR.mkdir(exist_ok=True)

# CLASS_NAMES = ['drone', 'airplane', 'helicopter']
# ID2LABEL    = {i: name for i, name in enumerate(CLASS_NAMES)}
# LABEL2ID    = {name: i for i, name in enumerate(CLASS_NAMES)}

CLASS_NAMES = ['drone', 'not_drone']
ID2LABEL    = {0: 'drone', 1: 'not_drone'}
LABEL2ID    = {'drone': 0, 'not_drone': 1}

MODEL_ID = 'PekingU/rtdetr_r50vd'

IMG_SIZE   = 640
BATCH_SIZE = 4      
GRAD_ACCUM = 4
NUM_EPOCHS = 45
LR         = 1e-4

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU:  Tesla T4
VRAM: 15.6 GB


In [15]:
def class_name_from_filename(filename: str) -> str:
    match = re.match(r'V_([A-Z]+)_', Path(filename).name)
    return match.group(1).lower() if match else 'unknown'

# def class_id_from_filename(filename: str) -> int:
#     return LABEL2ID.get(class_name_from_filename(filename), 0)
def class_id_from_filename(filename: str) -> int:
    name = class_name_from_filename(filename)  # 'drone' / 'airplane' / 'helicopter'
    return 0 if name == 'drone' else 1

In [16]:
def yolo_to_coco(box, img_w, img_h):
    x_c, y_c, w, h = box

    x_min = (x_c - w / 2) * img_w
    y_min = (y_c - h / 2) * img_h
    w     = w * img_w
    h     = h * img_h

    return [x_min, y_min, w, h]

In [17]:
def load_synthetic_to_coco(images_dir: Path, labels_dir: Path):
    images = []
    annotations = []

    ann_id = 0
    img_id = 0

    for img_path in sorted(images_dir.glob('*')):
        image = Image.open(img_path)
        w, h = image.size

        images.append({
            'id': img_id,
            'file_name': img_path.name,
            'width': w,
            'height': h,
        })

        label_path = labels_dir / (img_path.stem + '.txt')

        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    cls, x, y, bw, bh = map(float, line.strip().split())

                    bbox = yolo_to_coco((x, y, bw, bh), w, h)

                    annotations.append({
                        'id': ann_id,
                        'image_id': img_id,
                        'category_id': int(cls),
                        'bbox': bbox,
                        'area': bbox[2] * bbox[3],
                        'iscrowd': 0,
                    })

                    ann_id += 1

        img_id += 1

    return {
        'images': images,
        'annotations': annotations,
    }

In [18]:
def merge_coco(base, extra):
    max_img_id = max(img['id'] for img in base['images']) + 1
    max_ann_id = max(ann['id'] for ann in base['annotations']) + 1

    # shift images
    for img in extra['images']:
        img['id'] += max_img_id

    # shift annotations
    for ann in extra['annotations']:
        ann['id'] += max_ann_id
        ann['image_id'] += max_img_id

    return {
        **base,
        'images': base['images'] + extra['images'],
        'annotations': base['annotations'] + extra['annotations'],
    }

In [19]:
EXCLUDE_CLASSES = ['bird']

def filter_coco(split: str) -> dict:
    split_dir = DATA_ROOT / split
    json_path = next(split_dir.glob('*.json'))
    with open(json_path) as f:
        coco = json.load(f)

    excluded_ids = {
        img['id']
        for img in coco['images']
        if class_name_from_filename(img['file_name']) in EXCLUDE_CLASSES
    }

    clean_images = [img for img in coco['images']
                    if img['id'] not in excluded_ids]
    clean_anns   = [ann for ann in coco['annotations']
                    if ann['image_id'] not in excluded_ids]

    removed = len(coco['images']) - len(clean_images)
    print(f'[{split:>5}]  видалено: {removed:>3} | '
          f'залишилось: {len(clean_images):>5} зображень, '
          f'{len(clean_anns):>6} анотацій')

    return {**coco, 'images': clean_images, 'annotations': clean_anns}


clean_cocos = {split: filter_coco(split) for split in ['train', 'valid', 'test']}

SYNTH_DIR = Path('/kaggle/input/datasets/alinakharchenko1/synthetic/synthetic-drone-dataset-2k')

synthetic_coco = load_synthetic_to_coco(
    SYNTH_DIR / 'images',
    SYNTH_DIR / 'labels'
)

# print("after adding synthetic data:")
# clean_cocos['train'] = merge_coco(clean_cocos['train'], synthetic_coco)
# for split in SPLITS:
#     coco = clean_cocos[split]
#     print(f'[{split:>5}]  зображень: {len(coco["images"]):>5} | '
#           f'анотацій: {len(coco["annotations"]):>6}')

[train]  видалено: 1978 | залишилось:  8821 зображень,   8997 анотацій
[valid]  видалено: 109 | залишилось:   494 зображень,    497 анотацій
[ test]  видалено: 123 | залишилось:   473 зображень,    479 анотацій


In [20]:
print(len(clean_cocos['train']['images']))

8821


In [21]:
class SyntheticDroneDataset(Dataset):
    def __init__(self, images_dir: Path, coco_dict: dict, processor):
        self.images_dir = images_dir        # явно передаємо папку з картинками
        self.processor  = processor
        self.images     = coco_dict['images']
        self.img_map    = {img['id']: img for img in self.images}
        self.ann_map: dict[int, list] = {}
        for ann in coco_dict['annotations']:
            self.ann_map.setdefault(ann['image_id'], []).append(ann)

    def __len__(self):
        return len(self.images)

    def _find_image(self, file_name: str) -> Path:
        # шукаємо тільки в папці синтетики
        p = self.images_dir / file_name
        if p.exists():
            return p
        return next(self.images_dir.rglob(Path(file_name).name))

    def __getitem__(self, idx: int):
        meta   = self.images[idx]
        img_id = meta['id']

        try:
            image = Image.open(self._find_image(meta['file_name'])).convert('RGB')
        except (StopIteration, FileNotFoundError):
            # файл не знайдено → пропускаємо
            return self.__getitem__((idx + 1) % len(self))

        W_orig, H_orig = image.size
        anns = self.ann_map.get(img_id, [])
        if len(anns) == 0:
            return self.__getitem__((idx + 1) % len(self))

        # синтетика вже має правильний category_id з YOLO-конвертації
        boxes  = [ann['bbox'] for ann in anns]
        labels = [ann['category_id'] for ann in anns]

        target = {
            'image_id': img_id,
            'annotations': [
                {'bbox': b, 'category_id': l, 'iscrowd': 0, 'area': b[2]*b[3]}
                for b, l in zip(boxes, labels)
            ]
        }

        encoding = self.processor(images=image, annotations=target, return_tensors='pt')

        ret = {}
        for k, v in encoding.items():
            if torch.is_tensor(v):
                ret[k] = v.squeeze(0)
                if 'labels' in ret and 'class_labels' not in ret:
                    ret['class_labels'] = ret.pop('labels')
            else:
                labels_list = v
                for labels_dict in labels_list:
                    if 'labels' in labels_dict and 'class_labels' not in labels_dict:
                        labels_dict['class_labels'] = labels_dict.pop('labels')
                ret[k] = labels_list

        ret['orig_size'] = torch.tensor([H_orig, W_orig])
        ret['image_id']  = torch.tensor(img_id)
        return ret

In [22]:
class DroneDetectionDataset(Dataset):
    def __init__(self, split: str, processor, augment: bool = False):
        self.img_dir   = DATA_ROOT / split
        self.processor = processor
        self.augment   = augment

        coco = clean_cocos[split]

        self.images  = coco['images']
        self.img_map = {img['id']: img for img in self.images}

        self.ann_map: dict[int, list] = {}
        for ann in coco['annotations']:
            self.ann_map.setdefault(ann['image_id'], []).append(ann)

    def __len__(self):
        return len(self.images)

    def _find_image(self, file_name: str) -> Path:
        p = self.img_dir / file_name
        if p.exists():
            return p
        return next(self.img_dir.rglob(Path(file_name).name))

    def __getitem__(self, idx: int):
        meta   = self.images[idx]
        img_id = meta['id']
        image  = Image.open(self._find_image(meta['file_name'])).convert('RGB')

        anns   = self.ann_map.get(img_id, [])

        if len(anns) == 0:
            return self.__getitem__((idx + 1) % len(self))
        
        boxes  = [ann['bbox'] for ann in anns]

        label  = class_id_from_filename(meta['file_name'])
        labels = [label] * len(anns)

        target = {
            'image_id': img_id,
            'annotations': [
                {
                    'bbox'       : b,
                    'category_id': l,
                    'iscrowd'    : 0,
                    'area'       : b[2] * b[3],
                }
                for b, l in zip(boxes, labels)
            ]
        }

        encoding = self.processor(
            images=image,
            annotations=target,
            return_tensors='pt',
        )

        ret = {}
        for k, v in encoding.items():
            if torch.is_tensor(v):
                ret[k] = v.squeeze(0)
                
                if 'labels' in ret and 'class_labels' not in ret:
                    ret['class_labels'] = ret.pop('labels')
                
            else:
                labels_list = v

                for labels_dict in labels_list:
                    if 'labels' in labels_dict and 'class_labels' not in labels_dict:
                        labels_dict['class_labels'] = labels_dict.pop('labels')

                ret[k] = labels_list
        
        return ret

In [23]:
def collate_fn(batch):
    pixel_values = torch.stack([b['pixel_values'] for b in batch])
    
    labels = [
        b['labels'][0]
        for b in batch
    ]

    return {
        'pixel_values': pixel_values,
        'labels': labels
    }


processor = RTDetrImageProcessor.from_pretrained(
    MODEL_ID,
    size={'width': IMG_SIZE, 'height': IMG_SIZE},
)

train_ds = DroneDetectionDataset('train', processor, augment=True)
valid_ds = DroneDetectionDataset('valid', processor, augment=False)
test_ds  = DroneDetectionDataset('test',  processor, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                            shuffle=True,  collate_fn=collate_fn,
                            num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE,
                            shuffle=False, collate_fn=collate_fn,
                            num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):>5} зображень | {len(train_loader):>4} батчів')
print(f'Valid: {len(valid_ds):>5} зображень | {len(valid_loader):>4} батчів')
print(f'Test:  {len(test_ds):>5} зображень')

preprocessor_config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

Train:  8821 зображень | 2206 батчів
Valid:   494 зображень |  124 батчів
Test:    473 зображень


In [24]:
from torch.utils.data import ConcatDataset

synth_ds = SyntheticDroneDataset(
    images_dir = SYNTH_DIR / 'images',
    coco_dict  = synthetic_coco,
    processor  = processor,
)

# об'єднуємо реальний train + синтетика
combined_train_ds = ConcatDataset([train_ds, synth_ds])

print(f'Real:      {len(train_ds)}')
print(f'Synthetic: {len(synth_ds)}')
print(f'Combined:  {len(combined_train_ds)}')

train_loader = DataLoader(combined_train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_fn,
                          num_workers=2, pin_memory=True)

Real:      8821
Synthetic: 2000
Combined:  10821


In [25]:
model = RTDetrForObjectDetection.from_pretrained(
    MODEL_ID,
    num_labels=len(ID2LABEL),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
model.config.num_denoising = 0

model.to(DEVICE)

model = model.to('cuda:0')

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Параметрів всього:    {total/1e6:.1f}M')
print(f'Тренованих:           {trainable/1e6:.1f}M')
print(f'Класів:               {list(ID2LABEL.values())}')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/172M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/764 [00:00<?, ?it/s]

RTDetrForObjectDetection LOAD REPORT from: PekingU/rtdetr_r50vd
Key                                                 | Status   |                                                                                      
----------------------------------------------------+----------+--------------------------------------------------------------------------------------
model.decoder.class_embed.{0, 1, 2, 3, 4, 5}.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([80, 256]) vs model:torch.Size([2, 256])
model.enc_score_head.weight                         | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([80, 256]) vs model:torch.Size([2, 256])
model.decoder.class_embed.{0, 1, 2, 3, 4, 5}.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([80]) vs model:torch.Size([2])          
model.enc_score_head.bias                           | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([80]) vs model:torch.Size([2])          
model.denoising_class_embed.we

Параметрів всього:    42.7M
Тренованих:           42.7M
Класів:               ['drone', 'not_drone']


In [26]:
args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.05,
    weight_decay                = 1e-4,

    logging_dir                 = str(OUTPUT_DIR / 'logs'),
    logging_steps               = 20,
    eval_strategy               = 'steps',
    eval_steps                  = 500,
    save_strategy               = "steps",
    save_steps                  = 500,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',

    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 2,
    remove_unused_columns       = False,
    report_to                   = ['tensorboard'],
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [27]:
class DetectionTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(
            pixel_values=inputs['pixel_values'],
            labels=inputs['labels'],
        )
        return (outputs.loss, outputs) if return_outputs else outputs.loss

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix='eval'):
        metrics = super().evaluate(eval_dataset, ignore_keys, metric_key_prefix)
        metrics.update(self._compute_map(eval_dataset or self.eval_dataset))
        self.log(metrics)
        return metrics

    @torch.no_grad()
    def _compute_map(self, dataset) -> dict:
        self.model.eval()
        
        self.model.to(DEVICE)
        
        loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                            collate_fn=collate_fn, num_workers=0)

        coco_gt_dict = {
            'images'    : [],
            'annotations': [],
            'categories': [{'id': i, 'name': n} for i, n in ID2LABEL.items()],
        }
        coco_dt_list = []
        ann_id = 0

        for batch in loader:
            pixel_values = batch['pixel_values'].to(DEVICE)
            labels       = batch['labels']

            outputs = self.model(pixel_values=pixel_values)

            orig_sizes = torch.tensor(
                [[l['orig_size'][0], l['orig_size'][1]] for l in labels]
            ).to(DEVICE)

            results = processor.post_process_object_detection(
                outputs,
                threshold=0.3,
                target_sizes=orig_sizes,
            )

            for res, lbl in zip(results, labels):
                img_id = int(lbl['image_id'])
                H = int(lbl['orig_size'][0])
                W = int(lbl['orig_size'][1])

                coco_gt_dict['images'].append(
                    {'id': img_id, 'width': W, 'height': H}
                )

                for box, cls in zip(lbl['boxes'], lbl['class_labels']):
                    x1, y1, x2, y2 = box.tolist()
                    x1, y1, x2, y2 = x1*W, y1*H, x2*W, y2*H
                    coco_gt_dict['annotations'].append({
                        'id'         : ann_id,
                        'image_id'   : img_id,
                        'category_id': int(cls),
                        'bbox'       : [x1, y1, x2-x1, y2-y1],
                        'area'       : (x2-x1) * (y2-y1),
                        'iscrowd'    : 0,
                    })
                    ann_id += 1

                for score, cls, box in zip(
                    res['scores'].tolist(),
                    res['labels'].tolist(),
                    res['boxes'].tolist(),
                ):
                    x1, y1, x2, y2 = box
                    coco_dt_list.append({
                        'image_id'   : img_id,
                        'category_id': cls,
                        'bbox'       : [x1, y1, x2-x1, y2-y1],
                        'score'      : score,
                    })

        if not coco_dt_list:
            print('Жодного передбачення — threshold занадто високий?')
            return {'eval_mAP50_95': 0., 'eval_mAP50': 0.}

        with tempfile.NamedTemporaryFile(
            mode='w', suffix='.json', delete=False
        ) as f:
            json.dump(coco_gt_dict, f)
            gt_path = f.name

        coco_gt   = COCO(gt_path)
        coco_dt   = coco_gt.loadRes(coco_dt_list)
        coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()

        metrics = {
            'eval_mAP50_95' : round(coco_eval.stats[0], 4),
            'eval_mAP50'    : round(coco_eval.stats[1], 4),
            'eval_mAP_small': round(coco_eval.stats[3], 4),
            'eval_mAP_med'  : round(coco_eval.stats[4], 4),
            'eval_mAP_large': round(coco_eval.stats[5], 4),
        }

        # per-class AP50
        for cls_id, cls_name in ID2LABEL.items():
            coco_eval.params.catIds = [cls_id]
            coco_eval.evaluate()
            coco_eval.accumulate()
            metrics[f'eval_AP50_{cls_name}'] = round(coco_eval.stats[1], 4)

        return metrics

In [28]:
trainer = DetectionTrainer(
    model          = model,
    args           = args,
    train_dataset  = combined_train_ds,
    eval_dataset   = valid_ds,
    data_collator  = collate_fn,
)

2026-04-21 09:13:36.249847: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776762816.482054      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776762816.550733      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776762817.055759      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776762817.055803      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776762817.055806      55 computation_placer.cc:177] computation placer alr

In [29]:
class DetectionTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(
            pixel_values=inputs['pixel_values'],
            labels=inputs['labels'],
        )
        return (outputs.loss, outputs) if return_outputs else outputs.loss

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix='eval'):
        metrics = super().evaluate(eval_dataset, ignore_keys, metric_key_prefix)
        metrics.update(self._compute_map(eval_dataset or self.eval_dataset))
        self.log(metrics)
        return metrics

    @torch.no_grad()
    def _compute_map(self, dataset) -> dict:
        self.model.eval()
        
        self.model.to(DEVICE)
        
        loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                            collate_fn=collate_fn, num_workers=2)

        coco_gt_dict = {
            'images'    : [],
            'annotations': [],
            'categories': [{'id': i, 'name': n} for i, n in ID2LABEL.items()],
        }
        coco_dt_list = []
        ann_id = 0

        for batch in loader:
            pixel_values = batch['pixel_values'].to(DEVICE)
            labels       = batch['labels']

            outputs = self.model(pixel_values=pixel_values)

            orig_sizes = torch.tensor(
                [[l['orig_size'][0], l['orig_size'][1]] for l in labels]
            ).to(DEVICE)

            results = processor.post_process_object_detection(
                outputs,
                threshold=0.3,
                target_sizes=orig_sizes,
            )

            for res, lbl in zip(results, labels):
                img_id = int(lbl['image_id'])
                H = int(lbl['orig_size'][0])
                W = int(lbl['orig_size'][1])

                coco_gt_dict['images'].append(
                    {'id': img_id, 'width': W, 'height': H}
                )

                for box, cls in zip(lbl['boxes'], lbl['class_labels']):
                    x1, y1, x2, y2 = box.tolist()
                    x1, y1, x2, y2 = x1*W, y1*H, x2*W, y2*H
                    coco_gt_dict['annotations'].append({
                        'id'         : ann_id,
                        'image_id'   : img_id,
                        'category_id': int(cls),
                        'bbox'       : [x1, y1, x2-x1, y2-y1],
                        'area'       : (x2-x1) * (y2-y1),
                        'iscrowd'    : 0,
                    })
                    ann_id += 1

                for score, cls, box in zip(
                    res['scores'].tolist(),
                    res['labels'].tolist(),
                    res['boxes'].tolist(),
                ):
                    x1, y1, x2, y2 = box
                    coco_dt_list.append({
                        'image_id'   : img_id,
                        'category_id': cls,
                        'bbox'       : [x1, y1, x2-x1, y2-y1],
                        'score'      : score,
                    })

        if not coco_dt_list:
            print('Жодного передбачення — threshold занадто високий?')
            return {'eval_mAP50_95': 0., 'eval_mAP50': 0.}

        with tempfile.NamedTemporaryFile(
            mode='w', suffix='.json', delete=False
        ) as f:
            json.dump(coco_gt_dict, f)
            gt_path = f.name

        coco_gt   = COCO(gt_path)
        coco_dt   = coco_gt.loadRes(coco_dt_list)
        coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()

        metrics = {
            'eval_mAP50_95' : round(coco_eval.stats[0], 4),
            'eval_mAP50'    : round(coco_eval.stats[1], 4),
            'eval_mAP_small': round(coco_eval.stats[3], 4),
            'eval_mAP_med'  : round(coco_eval.stats[4], 4),
            'eval_mAP_large': round(coco_eval.stats[5], 4),
        }

        # per-class AP50
        for cls_id, cls_name in ID2LABEL.items():
            coco_eval.params.catIds = [cls_id]
            coco_eval.evaluate()
            coco_eval.accumulate()
            metrics[f'eval_AP50_{cls_name}'] = round(coco_eval.stats[1], 4)

        return metrics

In [30]:
ch_d = Path("/kaggle/input/datasets/alinakharchenko1/sent-checkpoints/rtdetr_drone_with_synt")

In [31]:
last_checkpoint = get_last_checkpoint(str(ch_d))

if last_checkpoint is not None:
    print(f"Знайдено чекпоінт: {last_checkpoint}")
else:
    print("Чекпоінтів не знайдено")

Знайдено чекпоінт: /kaggle/input/datasets/alinakharchenko1/sent-checkpoints/rtdetr_drone_with_synt/checkpoint-25000


In [32]:
ds = DroneDetectionDataset('train', processor)
print(len(ds))
print(ds[0])

8821
{'pixel_values': tensor([[[0.4471, 0.4471, 0.4471,  ..., 0.4392, 0.4392, 0.4392],
         [0.4471, 0.4471, 0.4471,  ..., 0.4392, 0.4392, 0.4392],
         [0.4471, 0.4471, 0.4471,  ..., 0.4392, 0.4392, 0.4392],
         ...,
         [0.5373, 0.5373, 0.5373,  ..., 0.5059, 0.5059, 0.5059],
         [0.5373, 0.5373, 0.5373,  ..., 0.5059, 0.5059, 0.5059],
         [0.5373, 0.5373, 0.5373,  ..., 0.5059, 0.5059, 0.5059]],

        [[0.5176, 0.5176, 0.5176,  ..., 0.5020, 0.5020, 0.5020],
         [0.5176, 0.5176, 0.5176,  ..., 0.5020, 0.5020, 0.5020],
         [0.5176, 0.5176, 0.5176,  ..., 0.5020, 0.5020, 0.5020],
         ...,
         [0.6000, 0.6000, 0.6000,  ..., 0.5765, 0.5765, 0.5765],
         [0.6000, 0.6000, 0.6000,  ..., 0.5765, 0.5765, 0.5765],
         [0.6000, 0.6000, 0.6000,  ..., 0.5765, 0.5765, 0.5765]],

        [[0.6118, 0.6118, 0.6118,  ..., 0.5922, 0.5922, 0.5922],
         [0.6118, 0.6118, 0.6118,  ..., 0.5922, 0.5922, 0.5922],
         [0.6118, 0.6118, 0.6118,  .

In [34]:
resume_from = None
if os.path.isdir(str(ch_d)):
    checkpoints = [d for d in os.listdir(str(ch_d)) if "checkpoint" in d]
    if checkpoints:
        resume_from = True

print('Training starting...')
trainer.train(resume_from_checkpoint="/kaggle/input/datasets/alinakharchenko1/sent-checkpoints/rtdetr_drone_with_synt/checkpoint-25000"
)

trainer.save_model(str(OUTPUT_DIR / 'final'))
processor.save_pretrained(str(OUTPUT_DIR / 'final'))
print('saved to', OUTPUT_DIR / 'final')

Training starting...


Step,Training Loss,Validation Loss


Exception in thread Thread-7 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_s

KeyboardInterrupt: 

In [ ]:
log_history = trainer.state.log_history
train_logs  = [x for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_logs   = [x for x in log_history if 'eval_loss' in x]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
ax = axes[0]
ax.plot([x['step']  for x in train_logs],
        [x['loss']  for x in train_logs], label='train loss')
ax.plot([x['step']  for x in eval_logs],
        [x['eval_loss'] for x in eval_logs], 'o-', label='val loss')
ax.set(xlabel='Step', ylabel='Loss', title='Train / Val Loss')
ax.legend()

# mAP
ax = axes[1]
if eval_logs and 'eval_mAP50' in eval_logs[0]:
    ax.plot([x['epoch'] for x in eval_logs],
            [x.get('eval_mAP50', 0)    for x in eval_logs], 'o-', label='mAP@50')
    ax.plot([x['epoch'] for x in eval_logs],
            [x.get('eval_mAP50_95', 0) for x in eval_logs], 's-', label='mAP@50:95')
    for cls_name in CLASS_NAMES:
        key = f'eval_AP50_{cls_name}'
        ax.plot([x['epoch'] for x in eval_logs],
                [x.get(key, 0) for x in eval_logs],
                '--', alpha=0.7, label=f'AP50 {cls_name}')
    ax.set(xlabel='Epoch', ylabel='mAP', title='mAP на валідації')
    ax.legend()

plt.tight_layout()
plt.show()

In [1]:
path_to_ready_model = "/kaggle/input/models/alinakharchenko1/synt-data-model/other/default/1"

In [20]:
processor = RTDetrImageProcessor.from_pretrained(path_to_ready_model)
model     = RTDetrForObjectDetection.from_pretrained(path_to_ready_model)
model.to(DEVICE)
model.eval()

print(f'Модель завантажена з {path_to_ready_model}')
print(f'Класів: {model.config.id2label}')

Loading weights:   0%|          | 0/763 [00:00<?, ?it/s]

RTDetrForObjectDetection LOAD REPORT from: /kaggle/input/models/alinakharchenko1/synt-data-model/other/default/1
Key                                | Status     |  | 
-----------------------------------+------------+--+-
model.denoising_class_embed.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель завантажена з /kaggle/input/models/alinakharchenko1/synt-data-model/other/default/1
Класів: {0: 'drone', 1: 'not_drone'}


In [21]:
def class_name_from_filename(filename: str) -> str:
    match = re.match(r'V_([A-Z]+)_', Path(filename).name)
    return match.group(1).lower() if match else 'unknown'

# def class_id_from_filename(filename: str) -> int:
#     return LABEL2ID.get(class_name_from_filename(filename), 0)
def class_id_from_filename(filename: str) -> int:
    name = class_name_from_filename(filename)  # 'drone' / 'airplane' / 'helicopter'
    return 0 if name == 'drone' else 1

EXCLUDE_CLASSES = ['bird']

def filter_coco(split: str) -> dict:
    split_dir = DATA_ROOT / split
    json_path = next(split_dir.glob('*.json'))
    with open(json_path) as f:
        coco = json.load(f)
    excluded_ids = {
        img['id'] for img in coco['images']
        if class_name_from_filename(img['file_name']) in EXCLUDE_CLASSES
    }
    clean_images = [img for img in coco['images'] if img['id'] not in excluded_ids]
    clean_anns   = [ann for ann in coco['annotations'] if ann['image_id'] not in excluded_ids]
    return {**coco, 'images': clean_images, 'annotations': clean_anns}

clean_cocos = {split: filter_coco(split) for split in ['test']}


class DroneDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, split: str, processor):
        self.img_dir  = DATA_ROOT / split
        self.processor = processor
        coco          = clean_cocos[split]
        self.images   = coco['images']
        self.img_map  = {img['id']: img for img in self.images}
        self.ann_map: dict[int, list] = {}
        for ann in coco['annotations']:
            self.ann_map.setdefault(ann['image_id'], []).append(ann)

    def __len__(self):
        return len(self.images)

    def _find_image(self, file_name: str) -> Path:
        p = self.img_dir / file_name
        if p.exists():
            return p
        return next(self.img_dir.rglob(Path(file_name).name))

    def __getitem__(self, idx: int):
        meta   = self.images[idx]
        img_id = meta['id']
        image  = Image.open(self._find_image(meta['file_name'])).convert('RGB')
        W_orig, H_orig = image.size
    
        anns = self.ann_map.get(img_id, [])
        if len(anns) == 0:
            return self.__getitem__((idx + 1) % len(self))
    
        boxes  = [ann['bbox'] for ann in anns]
        label  = class_id_from_filename(meta['file_name'])
        labels = [label] * len(anns)
        target = {
            'image_id': img_id,
            'annotations': [
                {'bbox': b, 'category_id': l, 'iscrowd': 0, 'area': b[2]*b[3]}
                for b, l in zip(boxes, labels)
            ]
        }
    
        encoding = self.processor(images=image, annotations=target, return_tensors='pt')
    
        ret = {}
        for k, v in encoding.items():
            if torch.is_tensor(v):
                ret[k] = v.squeeze(0)
                if 'labels' in ret and 'class_labels' not in ret:
                    ret['class_labels'] = ret.pop('labels')
            else:
                labels_list = v
                for labels_dict in labels_list:
                    if 'labels' in labels_dict and 'class_labels' not in labels_dict:
                        labels_dict['class_labels'] = labels_dict.pop('labels')
                ret[k] = labels_list
    
        ret['orig_size'] = torch.tensor([H_orig, W_orig])
        ret['image_id']  = torch.tensor(img_id)
    
        return ret


def collate_fn(batch):
    pixel_values = torch.stack([b['pixel_values'] for b in batch])
    labels = [{k: v for k, v in b.items() if k != 'pixel_values'} for b in batch]
    return {'pixel_values': pixel_values, 'labels': labels}


test_ds     = DroneDetectionDataset('test', processor)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                         shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f'Test: {len(test_ds)} зображень | {len(test_loader)} батчів')

Test: 473 зображень | 119 батчів


In [22]:
@torch.no_grad()
def evaluate_map(loader) -> dict:
    model.eval()
    coco_gt_dict = {
        'images'     : [],
        'annotations': [],
        'categories' : [{'id': i, 'name': n} for i, n in ID2LABEL.items()],
    }
    coco_dt_list = []
    ann_id = 0

    for batch in loader:
        pixel_values = batch['pixel_values'].to(DEVICE)
        labels       = batch['labels']

        outputs    = model(pixel_values=pixel_values)
        orig_sizes = torch.tensor(
            [[l['orig_size'][0], l['orig_size'][1]] for l in labels]
        ).to(DEVICE)
        results = processor.post_process_object_detection(
            outputs, threshold=0.5, target_sizes=orig_sizes
        )

        for res, lbl in zip(results, labels):
            img_id = int(lbl['image_id'])
            H      = int(lbl['orig_size'][0])
            W      = int(lbl['orig_size'][1])
        
            # boxes і class_labels знаходяться всередині lbl['labels'][0]
            inner  = lbl['labels'][0] if isinstance(lbl.get('labels'), list) else lbl
        
            coco_gt_dict['images'].append({'id': img_id, 'width': W, 'height': H})
        
            for box, cls in zip(inner['boxes'], inner['class_labels']):
                cx, cy, w, h = box.tolist()
                abs_w = w * W
                abs_h = h * H
                abs_x = cx * W - abs_w / 2
                abs_y = cy * H - abs_h / 2
                coco_gt_dict['annotations'].append({
                    'id'         : ann_id,
                    'image_id'   : img_id,
                    'category_id': int(cls),
                    'bbox'       : [abs_x, abs_y, abs_w, abs_h],
                    'area'       : abs_w * abs_h,
                    'iscrowd'    : 0,
                })
                ann_id += 1
        
            for score, cls, box in zip(
                res['scores'].tolist(),
                res['labels'].tolist(),
                res['boxes'].tolist(),
            ):
                x1, y1, x2, y2 = box
                coco_dt_list.append({
                    'image_id'   : img_id,
                    'category_id': cls,
                    'bbox'       : [x1, y1, x2-x1, y2-y1],
                    'score'      : score,
                })

    if not coco_dt_list:
        print('Жодного передбачення!')
        return {}

    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        json.dump(coco_gt_dict, f)
        gt_path = f.name

    coco_gt   = COCO(gt_path)
    coco_dt   = coco_gt.loadRes(coco_dt_list)
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    coco_eval.evaluate(); coco_eval.accumulate(); coco_eval.summarize()

    metrics = {
        'mAP@50:95' : round(coco_eval.stats[0], 4),
        'mAP@50'    : round(coco_eval.stats[1], 4),
        'mAP@75'    : round(coco_eval.stats[2], 4),
        'mAP_small' : round(coco_eval.stats[3], 4),
        'mAP_medium': round(coco_eval.stats[4], 4),
        'mAP_large' : round(coco_eval.stats[5], 4),
    }

    # per-class AP50
    for cls_id, cls_name in ID2LABEL.items():
        coco_eval.params.catIds = [cls_id]
        coco_eval.evaluate(); coco_eval.accumulate()
        metrics[f'AP50_{cls_name}'] = round(coco_eval.stats[1], 4)

    return metrics, coco_gt_dict, coco_dt_list


metrics, coco_gt_dict, coco_dt_list = evaluate_map(test_loader)

print('\nTEST RESULTS')
for k, v in metrics.items():
    print(f'  {k:<20} {v:.4f}')

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.13s).
Accumulating evaluation results...
DONE (t=0.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.635
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.958
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.668
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.601
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.630
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.832
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.691
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.698
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

TEST RESULTS - detection (with synt)

  mAP@50:95            0.6355
  
  mAP@50               0.9578
  
  mAP@75               0.6676
  
  mAP_small            0.6013
  
  mAP_medium           0.6304
  
  mAP_large            0.8316
  
  AP50_drone           0.9578
  
  AP50_not_drone       0.9578

TEST RESULTS - detection (no synt)

  mAP@50:95            0.6326
  
  mAP@50               0.9532
  
  mAP@75               0.6803
  
  mAP_small            0.5904
  
  mAP_medium           0.6329
  
  mAP_large            0.8297
  
  AP50_drone           0.9532
  
  AP50_not_drone       0.9532

=== TEST RESULTS === classification

  mAP@50:95            0.6831
  
  mAP@50               0.9765
  
  mAP@75               0.7638
  
  mAP_small            0.5903
  
  mAP_medium           0.6788
  
  mAP_large            0.9178
  
  AP50_drone           0.9765
  
  AP50_airplane        0.9765
  
  AP50_helicopter      0.9765

In [ ]:
# batch = next(iter(test_loader))
# lbl = batch['labels'][0]
# print('Ключі в lbl:', lbl.keys())
# for k, v in lbl.items():
#     if torch.is_tensor(v):
#         print(f'  {k}: tensor shape={v.shape}, dtype={v.dtype}')
#     else:
#         print(f'  {k}: {type(v)} = {v}')